# Aspect 1 — Does Message Direction Matter in Relational GNNs?

Relational databases encode FK→PK relationships that give edges a natural direction (child → parent).
We ask: **does message-passing direction affect node classification on relational graphs?**

We test three modes across two architectures and two real-world datasets:

| Mode | Description |
|------|-------------|
| **MPNN-U** | Undirected — messages flow on all edges (FK→PK + reversed PK→FK) |
| **MPNN-D** | Directed — messages flow only on FK→PK edges (child → parent) |
| **Dir-GNN** | Bidirectional with separate parameters per direction |

**Hypothesis:** Since both tasks are user-level, signal should aggregate naturally upward via FK→PK edges (child rows → user). MPNN-D may match or exceed MPNN-U by avoiding reverse noise; Dir-GNN may do best by separating the two directions.

## 1. Setup

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch

# Resolve ROOT to aspect1/ regardless of where the kernel was started
_here = Path(os.path.abspath(""))
ROOT  = _here if _here.name == "aspect1" else _here / "aspect1"
os.chdir(ROOT)  # keep CWD in sync so relative subprocess calls work

PROCESSED   = ROOT / "processed"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS_CSV = ROOT / "results" / "metrics.csv"
PYTHON      = sys.executable

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"ROOT:   {ROOT}")
print(f"Device: {DEVICE}")
print(f"Python: {PYTHON}")

# ── Set to True to skip preprocessing & training and jump straight to results ──
# Useful when sharing the notebook after training is done on a server.
RESULTS_ONLY = False

## 2. Datasets

We use two [RelBench](https://relbench.stanford.edu/) datasets, both with binary user-level classification targets.

### rel-stack / user-engagement
**Source:** Stack Exchange Q&A platform.  
**Task:** Predict whether a user will receive a *positive* vote in the next 2 years.  
**Schema highlights:**
- **users** (target) → posts, comments, votes, badges, postHistory
- 7 node types, 11 FK→PK edges
- FK direction: child rows (posts, comments, …) reference parent (users)

### rel-avito / user-visits
**Source:** Avito — Russian online classifieds platform.  
**Task:** Predict whether a user will visit an ad in the next 2 weeks.  
**Schema highlights:**
- **UserInfo** (target) → VisitStream, SearchStream, PhoneRequestsStream, SearchInfo, AdsInfo
- 8 node types, 11 FK→PK edges
- Similar user-level structure to rel-stack

**Directionality rationale:** For user-level tasks, the natural signal flow is FK→PK (child activity → user node). MPNN-D (directed) is designed to capture exactly this; MPNN-U also receives reversed edges (parent → child), which could add noise or complementary context.

In [ ]:
TASKS = [
    ("rel-stack", "user-engagement"),
    ("rel-avito",  "user-visits"),
]

for dataset, task in TASKS:
    meta_path = PROCESSED / dataset / task / "meta.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        fwd = len(meta["fwd_edge_types"])
        rev = len(meta["rev_edge_types"])
        ntypes = len(meta["node_types"])
        feat_dims = meta["node_feat_dims"]
        print(f"\n{dataset}/{task}")
        print(f"  Target node : {meta['target_node']}")
        print(f"  Node types  : {ntypes}  ({', '.join(meta['node_types'])})")
        print(f"  Edges       : {fwd} FK→PK + {rev} rev PK→FK = {fwd+rev} total")
        print(f"  Feature dims: {feat_dims}")
    else:
        print(f"{dataset}/{task}: not yet preprocessed")

## 3. Preprocessing

The preprocessing script (`preprocess.py`) builds `train.pt`, `val.pt`, `test.pt`, and `meta.json`
for each dataset/task. It:
- Downloads the RelBench dataset via `relbench.get_dataset()` (cached locally after first run)
- Applies temporal splits using `db.upto(cutoff)` — train/val share the same cutoff timestamp
- Encodes numerical features with `StandardScaler`, categoricals with `OrdinalEncoder` (fitted on train only)
- Builds a `HeteroData` graph with FK→PK edges plus reversed PK→FK edges (prefixed `rev_`)
- Drops leakage columns from the task's `remove_columns` list

**If `.pt` files already exist, this cell is skipped.**

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping preprocessing.")
else:
    def preprocessing_done():
        return all(
            (PROCESSED / ds / task / "meta.json").exists()
            for ds, task in TASKS
        )

    if preprocessing_done():
        print("Preprocessed data found — skipping preprocessing.")
    else:
        print("Running preprocess.py (this may take 20-40 minutes on first run) …")
        result = subprocess.run(
            [PYTHON, "preprocess.py"],
            cwd=ROOT,
            capture_output=False,
        )
        if result.returncode != 0:
            raise RuntimeError("preprocess.py failed — check output above.")
        print("\nPreprocessing complete.")

In [ ]:
# Show graph statistics from the preprocessed .pt files
for dataset, task in TASKS:
    meta_path = PROCESSED / dataset / task / "meta.json"
    if not meta_path.exists():
        continue
    with open(meta_path) as f:
        meta = json.load(f)
    target = meta["target_node"]
    print(f"\n{'='*50}")
    print(f"{dataset} / {task}")
    print(f"{'='*50}")
    for split in ["train", "val", "test"]:
        pt = PROCESSED / dataset / task / f"{split}.pt"
        hdata = torch.load(pt, weights_only=False, map_location="cpu")
        n_nodes = hdata[target].num_nodes
        n_labeled = int(hdata[target].mask.sum())
        n_pos = int(hdata[target].y[hdata[target].mask].sum())
        del hdata
        print(f"  {split:5s}: {n_nodes:6,} nodes  {n_labeled:6,} labeled  "
              f"{n_pos:5,} positive  ({100*n_pos/max(n_labeled,1):.1f}%)")

## 4. Model Architectures

All models share the same backbone:
- Lazy initialization (`(-1, -1)`) in SAGEConv/GATConv handles varying feature dimensions across node types
- `hidden_dim = 64`, `dropout = 0.3`
- Binary head: `Linear(64 → 1) + Sigmoid`
- `to_hetero()` lifts homogeneous convolutions to heterogeneous graphs

### MPNN-U (Undirected)
```
All edges (FK→PK ∪ rev PK→FK) → to_hetero(SAGEConv/GATConv) × L layers
```
Messages flow in both directions. Child rows push features to parent users, and vice versa.

### MPNN-D (Directed, FK→PK only)
```
FK→PK edges only → to_hetero(SAGEConv/GATConv) × L layers
```
Messages flow only from child rows up to parent nodes. Reverse edges are filtered out at forward-pass time.

### Dir-GNN (Bidirectional with Separate Parameters)
```
Per layer:
  fwd_out = to_hetero(Conv(hidden//2), fwd_edges)   # FK→PK
  rev_out = to_hetero(Conv(hidden//2), rev_edges)   # PK→FK
  x = Linear(cat([fwd_out, rev_out])) + ReLU        # project back to hidden
```
Each layer runs two separate convolutions (with independent parameters) then concatenates and projects. This is the full Dir-GNN formulation from Rossi et al. (2023).

In [ ]:
# Quick parameter count for each architecture × mode × layer combination
# (uses one dataset's metadata as a representative graph)
import sys; sys.path.insert(0, str(ROOT))
from models import build_model, count_parameters
from train import load_meta, load_split, make_loader, DEVICE as _DEV

_dataset, _task = "rel-stack", "user-engagement"
_meta = load_meta(_dataset, _task)
_data = load_split(_dataset, _task, "train")
_metadata = _data.metadata()
_target = _meta["target_node"]

print(f"{'Arch':6}  {'Mode':8}  {'L':2}  {'Params':>10}")
print("-" * 32)
for arch in ["sage", "gat"]:
    for mode in ["mpnn_u", "mpnn_d", "dir_gnn"]:
        for L in [1, 2, 3]:
            m = build_model(_metadata, _target, arch=arch, mode=mode,
                            num_layers=L, hidden_dim=64).to(_DEV)
            # warm-up lazy init
            _ldr = make_loader(_data, _target, "mask", 32, 1, L, False)
            _b = next(iter(_ldr)).to(_DEV)
            import torch
            with torch.no_grad():
                m({nt: _b[nt].x for nt in _b.node_types}, _b.edge_index_dict)
            n = count_parameters(m)
            print(f"{arch:6}  {mode:8}  {L:2}  {n:10,}")
    print()

## 5. Experiment Configuration

We run **36 experiments**: 2 datasets × 2 architectures × 3 modes × 3 depth levels × 1 seed.

| Axis | Values |
|------|--------|
| Dataset/Task | rel-stack/user-engagement, rel-avito/user-visits |
| Architecture | GraphSAGE, GAT |
| Mode | MPNN-U, MPNN-D, Dir-GNN |
| Layers | 1, 2, 3 |
| Seed | 0 |

Training details:
- Adam optimizer, lr=1e-3, weight_decay=1e-5
- Max 100 epochs with **early stopping** (patience=10) on val AUPRC
- Batch size 512 with NeighborLoader (10 neighbors per layer)
- Metric: **AUPRC** (area under precision-recall curve) — appropriate for class-imbalanced binary tasks

Checkpoints are saved to `checkpoints/{dataset}/{task}/{arch}_{mode}_L{layers}_s{seed}.pt`.
**Already-completed runs are skipped automatically.**

In [ ]:
ARCHS  = ["sage", "gat"]
MODES  = ["mpnn_u", "mpnn_d", "dir_gnn"]
LAYERS = [1, 2, 3]
SEED   = 0

def ckpt_path(dataset, task, arch, mode, num_layers, seed):
    return CHECKPOINTS / dataset / task / f"{arch}_{mode}_L{num_layers}_s{seed}.pt"

configs = [
    dict(dataset=ds, task=task, arch=arch, mode=mode, num_layers=L, seed=SEED)
    for ds, task in TASKS
    for arch in ARCHS
    for mode in MODES
    for L in LAYERS
]

done    = [c for c in configs if ckpt_path(**c).exists()]
pending = [c for c in configs if not ckpt_path(**c).exists()]
print(f"Total: {len(configs)}   Done: {len(done)}   Pending: {len(pending)}")
if pending:
    print("\nPending:")
    for c in pending:
        print(f"  {c['dataset']}/{c['task']}  {c['arch']}  {c['mode']}  L={c['num_layers']}")

## 6. Training

The cell below runs all pending configurations. Completed ones are automatically skipped.
Estimated time per run: ~2-5 min on GPU. Total wall time for 36 fresh runs: ~1.5-3 hours.

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping training.")
elif not pending:
    print("All runs complete — skipping training.")
else:
    print(f"Running {len(pending)} experiment(s) …")
    result = subprocess.run(
        [PYTHON, "-u", "run_experiments.py"],
        cwd=ROOT,
        capture_output=False,
    )
    if result.returncode != 0:
        raise RuntimeError("run_experiments.py failed — check output above.")

if not RESULTS_ONLY:
    # Re-check
    done    = [c for c in configs if ckpt_path(**c).exists()]
    pending = [c for c in configs if not ckpt_path(**c).exists()]
    print(f"\nAfter training: {len(done)}/{len(configs)} complete")
    if pending:
        print("Still pending (may have failed):")
        for c in pending:
            print(f"  {c['dataset']}/{c['task']}  {c['arch']}  {c['mode']}  L={c['num_layers']}")

## 7. Results

We load metrics from `results/metrics.csv` — one row per completed run — and analyze:
1. Main comparison: mode (MPNN-U vs MPNN-D vs Dir-GNN) by dataset and architecture
2. Effect of depth (L=1,2,3) per mode
3. Test AUPRC heatmap

In [ ]:
if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"No results file at {RESULTS_CSV}. Run training first.")

df = pd.read_csv(RESULTS_CSV)
print(f"Loaded {len(df)} rows from metrics.csv")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
df_all = df.drop_duplicates(
    subset=["dataset", "task", "mode", "arch", "num_layers", "seed"],
    keep="last"
).reset_index(drop=True)

MODE_LABELS = {"mpnn_u": "MPNN-U", "mpnn_d": "MPNN-D", "dir_gnn": "Dir-GNN"}
df_all["Mode"]    = df_all["mode"].map(MODE_LABELS)
df_all["Arch"]    = df_all["arch"].str.upper()
df_all["Dataset"] = df_all["dataset"] + "/" + df_all["task"]

print(f"Experiment rows: {len(df_all)}  (expecting up to 36)")
print(df_all.groupby(["Dataset", "Arch", "Mode"])["test_auprc"].mean().round(4).to_string())

In [ ]:
# ── Figure 1: Test AUPRC by Mode, Dataset, Architecture ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
datasets  = df_all["Dataset"].unique()
archs     = ["SAGE", "GAT"]
modes     = ["MPNN-U", "MPNN-D", "Dir-GNN"]
colors    = ["#4C72B0", "#DD8452", "#55A868"]
x         = np.arange(len(modes))
width     = 0.35

for ax, dataset in zip(axes, datasets):
    sub = df_all[df_all["Dataset"] == dataset]
    for i, arch in enumerate(archs):
        arch_sub = sub[sub["Arch"] == arch]
        vals = [
            arch_sub[arch_sub["Mode"] == m]["test_auprc"].mean()
            for m in modes
        ]
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=arch,
                      color=colors,
                      alpha=0.85 if arch == "SAGE" else 0.5,
                      edgecolor="white")
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, v + 0.002,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(modes)
    ax.set_title(dataset, fontsize=11, fontweight="bold")
    ax.set_ylabel("Test AUPRC (avg over L=1,2,3)")
    ax.set_ylim(bottom=0)
    ax.legend(title="Architecture")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Figure 1: Test AUPRC by Directionality Mode", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig1_mode_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 2: Effect of Depth (layers) per Mode ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
layer_vals  = [1, 2, 3]
mode_colors = dict(zip(modes, colors))

for ax, dataset in zip(axes, datasets):
    sub = df_all[df_all["Dataset"] == dataset]
    for mode in modes:
        mode_sub = sub[sub["Mode"] == mode]
        ys = [
            mode_sub[mode_sub["num_layers"] == L]["test_auprc"].mean()
            for L in layer_vals
        ]
        ax.plot(layer_vals, ys, marker="o", label=mode,
                color=mode_colors[mode], linewidth=2)
    ax.set_xticks(layer_vals)
    ax.set_xlabel("Number of Layers")
    ax.set_ylabel("Test AUPRC (avg over both archs)")
    ax.set_title(dataset, fontsize=11, fontweight="bold")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Figure 2: Effect of Depth on Test AUPRC", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig2_depth_effect.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 3: Heatmap — Mode × Dataset, avg test AUPRC ───────────────────────
pivot = df_all.groupby(["Dataset", "Mode"])["test_auprc"].mean().unstack("Mode")
pivot = pivot[["MPNN-U", "MPNN-D", "Dir-GNN"]]

fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=10)
plt.colorbar(im, ax=ax, label="Avg Test AUPRC")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                    color="black", fontsize=10, fontweight="bold")
ax.set_title("Figure 3: Avg Test AUPRC (all archs & depths)", fontsize=11)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig3_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Table 1: Full results summary ─────────────────────────────────────────────
summary = (
    df_all.groupby(["Dataset", "Arch", "Mode", "num_layers"])
    [["val_auprc", "test_auprc", "test_auc", "train_time_sec"]]
    .mean()
    .round(4)
)
summary.columns = ["Val AUPRC", "Test AUPRC", "Test AUC", "Train Time (s)"]
print("Full Results Table")
print(summary.to_string())

In [ ]:
# ── Best configuration per dataset ────────────────────────────────────────────
print("Best configuration per dataset (by test AUPRC):\n")
for dataset in datasets:
    sub = df_all[df_all["Dataset"] == dataset]
    if sub.empty:
        continue
    best = sub.loc[sub["test_auprc"].idxmax()]
    print(f"{dataset}")
    print(f"  Best:  {best['Arch']} / {best['Mode']} / L={best['num_layers']}")
    print(f"  Test AUPRC: {best['test_auprc']:.4f}   Test AUC: {best['test_auc']:.4f}")
    print()

## 8. Analysis & Conclusions

### What we tested
We evaluated three message-passing modes — **MPNN-U** (undirected), **MPNN-D** (FK→PK only), and **Dir-GNN** (bidirectional, separate parameters) — on two user-level relational tasks using GraphSAGE and GAT backbones at depths 1, 2, and 3.

### Key findings

*(This section is filled in after running the experiments. The analysis below describes the expected pattern; update after reviewing Figures 1–3.)*

**Mode comparison (Figure 1):**  
Since both tasks are user-level, child entities (posts, comments, visits) are the FK-side nodes and users are the PK-side target. We expect that FK→PK messages (child → user) carry the relevant signal, suggesting MPNN-D or Dir-GNN should match or exceed MPNN-U.
- If **MPNN-D ≈ MPNN-U**: reverse edges (parent → child) carry little additional signal; directionality mostly a red herring for this task type
- If **Dir-GNN > MPNN-D**: distinguishing the two directions (with separate parameters) helps even when one direction dominates
- If **MPNN-U > MPNN-D**: reverse edges carry useful global context (e.g., post popularity flowing back to users)

**Depth effect (Figure 2):**  
More layers expand the receptive field, but relational graphs can be dense. If performance peaks at L=2, deeper models may over-smooth.

**Architecture:**  
GAT's attention mechanism can weight neighbors selectively, which may be more beneficial than SAGE's mean aggregation when neighbor heterogeneity is high.

### Relation to the directionality hypothesis
Both rel-stack and rel-avito have user-level targets, so FK→PK edges naturally carry the relevant signal. A meaningful future extension would include a task where the target is on the FK side (child node), which would reverse the expected ordering and provide a stronger test of the directionality hypothesis.